In [3]:
# =========================
# CELL 1: Install dependencies
# =========================
!pip install -q groq chromadb sentence-transformers


In [9]:
# Kaggle install cell
!pip -q install -U transformers accelerate sentencepiece

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# Model name (gated Meta model)
PARAPHRASE_MODEL_NAME = "meta-llama/Llama-3.2-3B"

# Put your Hugging Face token here
HF_TOKEN = "HF_TOKEN"

_paraphrase_tokenizer = None
_paraphrase_model = None

def load_paraphrase_model():
    global _paraphrase_tokenizer, _paraphrase_model

    if _paraphrase_tokenizer is None or _paraphrase_model is None:
        print("Loading tokenizer...")
        _paraphrase_tokenizer = AutoTokenizer.from_pretrained(
            PARAPHRASE_MODEL_NAME,
            token=HF_TOKEN
        )
        if _paraphrase_tokenizer.pad_token is None:
            _paraphrase_tokenizer.pad_token = _paraphrase_tokenizer.eos_token

        print("Loading model (this may take time)...")
        _paraphrase_model = AutoModelForCausalLM.from_pretrained(
            PARAPHRASE_MODEL_NAME,
            token=HF_TOKEN,
            torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
            device_map="auto",
        )

    return _paraphrase_tokenizer, _paraphrase_model


In [11]:
# =========================
# CELL 2: Imports + API setup
# =========================
import os
import json
import time
import chromadb

from groq import Groq
from kaggle_secrets import UserSecretsClient
from sentence_transformers import CrossEncoder
from typing import TypedDict, List

user_secrets = UserSecretsClient()
groq_api_key = user_secrets.get_secret("GROQ_API_KEY")
groq_client = Groq(api_key=groq_api_key)

print("Groq client initialized.")


Groq client initialized.


In [12]:
from typing import TypedDict, List

class LabState(TypedDict):
    original_query: str
    route: str
    target_tags: List[str]
    expanded_query: str
    hyde_document: str
    expanded_query_chunks: List[dict]
    retrieved_chunks: List[dict]
    final_retrieved_chunks: List[dict]
    final_answer: str
    final_answer_raw: str


In [13]:
# =========================
# CELL 4: Director node
# =========================
import json

def chief_director(state: LabState) -> dict:
    try:
        response = groq_client.chat.completions.create(
            model="llama-3.1-8b-instant",
            response_format={"type": "json_object"},
            messages=[
                {
                    "role": "system",
                    "content": (
                        "You are the routing director for a Physical Vapor Deposition (PVD) AI Copilot. "
                        "Your job is to classify the user's query into a route and assign metadata tags.\n\n"

                        "STEP 1: Decide the route ('chat' or 'database').\n"
                        "Differentiate them like this:\n"
                        "- Route to 'chat' ONLY if the message is purely conversational and does not require any scientific, technical, experimental, or literature knowledge.\n"
                        "- Examples of 'chat': 'hello', 'hi', 'how are you', 'thanks', 'okay', 'good morning'.\n"
                        "- Route to 'database' for ANY query involving science, materials, physics, chemistry, engineering, PVD, sputtering, evaporation, thin films, deposition parameters, substrates, targets, plasma, vacuum, crystallinity, morphology, characterization tools, performance, interpretation, or lab methods.\n"
                        "- If a query mixes small talk with science, science takes priority and you MUST route to 'database'.\n"
                        "- If the user asks for papers, notes, authors, mechanisms, comparisons, explanations, process conditions, or interpretation in a scientific context, route to 'database'.\n"
                        "- If you are uncertain, prefer 'database' over 'chat'.\n\n"

                        "STEP 2: Assign tags.\n"
                        "- If the route is 'chat', the target_tags array MUST be empty [].\n"
                        "- If the route is 'database', read the ENTIRE query and classify it into an array of applicable tags. You MUST ONLY select from these exact four strings:\n"
                        "  1. 'Background': For theory, historical context, fundamental physics, and rationale.\n"
                        "  2. 'Synthesis': For recipes, deposition parameters (temperature, pressure), hardware, and fabrication steps.\n"
                        "  3. 'Characterization': For analytical tools (XRD, SEM, TEM), sample preparation, and physical property observations.\n"
                        "  4. 'Analysis': For performance data, electrochemical impedance (EIS), catalytic activity, and results interpretation.\n\n"

                        "CRITICAL RULES:\n"
                        "- DO NOT invent or hallucinate new tags. Only use the exact strings provided above.\n"
                        "- A query may have multiple tags if multiple intents are present.\n"
                        "- If the database query is too general and doesn't fit a specific category, return an empty array [].\n"
                        "- If the user mentions sample preparation, use 'Characterization'.\n"
                        "- If the user mentions deposition conditions, recipes, hardware, or process steps, use 'Synthesis'.\n"
                        "- If the user mentions results, impedance, performance, or interpretation, use 'Analysis'.\n"
                        "- If the user asks about theory or rationale, use 'Background'.\n\n"

                        'Return strict JSON with EXACTLY this structure: '
                        '{"reasoning": "string", "decision": "chat" | "database", "target_tags": ["Background"] | []}'
                    ),
                },
                {
                    "role": "user",
                    "content": state["original_query"],
                },
            ],
        )

        content = response.choices[0].message.content
        parsed = json.loads(content)

        return {
            "route": parsed.get("decision", "database"),
            "target_tags": parsed.get("target_tags", []),
        }

    except Exception as error:
        print(f"chief_director error: {error}")
        return {
            "route": "database",
            "target_tags": [],
        }


In [14]:
# =========================
# CELL 5: Query expander node
# =========================
import json

def query_expander(state: LabState) -> dict:
    try:
        response = groq_client.chat.completions.create(
            model="llama-3.1-8b-instant",
            response_format={"type": "json_object"},
            messages=[
                {
                    "role": "system",
                    "content": (
                        "You are a Material Science semantic translator for a Physical Vapor Deposition (PVD) AI Copilot. "
                        "Your task is to transform the user's raw query into a dense, keyword-rich semantic search string "
                        "optimized for a vector database.\n\n"
                        "You will receive:\n"
                        "1. The user's original query\n"
                        "2. Metadata tags assigned by the director agent\n\n"
                        "Your job:\n"
                        "- Preserve the user's actual scientific intent.\n"
                        "- Expand short or informal wording into formal academic terminology.\n"
                        "- Add closely related technical synonyms that would help retrieve relevant research passages.\n"
                        "- Expand abbreviations into full forms whenever useful.\n"
                        "- Include PVD-specific vocabulary only when relevant to the user's query and assigned tags.\n"
                        "- Use the assigned tags to bias the expansion toward the correct technical dimension.\n"
                        "- Do NOT answer the question.\n"
                        "- Do NOT explain your reasoning.\n"
                        "- Do NOT invent unrelated concepts.\n\n"
                        "Tag guidance:\n"
                        "- If 'Background' is present, emphasize theory, mechanisms, first principles, thermodynamics, kinetics, surface science, nucleation, growth mechanisms, plasma-material interaction, and historical/fundamental context.\n"
                        "- If 'Synthesis' is present, emphasize deposition recipes, process parameters, chamber conditions, substrate temperature, working pressure, gas flow, power, target composition, deposition rate, sputtering conditions, evaporation conditions, bias voltage, vacuum level, and fabrication workflow.\n"
                        "- If 'Characterization' is present, emphasize X-ray diffraction, XRD, scanning electron microscopy, SEM, transmission electron microscopy, TEM, atomic force microscopy, AFM, Raman spectroscopy, XPS, EDS/EDX, profilometry, ellipsometry, sample preparation, microstructure, morphology, phase identification, grain size, roughness, film thickness, and measured physical properties.\n"
                        "- If 'Analysis' is present, emphasize electrochemical impedance spectroscopy, EIS, impedance, conductivity, resistivity, carrier transport, catalytic activity, reaction kinetics, overpotential, current density, stability, durability, device performance, comparative trends, and interpretation of results.\n\n"
                        "Expansion rules:\n"
                        "- If the query contains a material name, chemical formula, substrate, gas, or process variable, keep it and expand around it.\n"
                        "- If the query is broad, produce a richer academic search phrase rather than a direct sentence answer.\n"
                        "- If multiple tags are present, combine the relevant vocabularies into one coherent retrieval query.\n"
                        "- Prefer terminology that would appear in research papers, reviews, and experimental sections.\n"
                        "- Keep the output concise enough for retrieval, but information-dense.\n\n"
                        'Return strict JSON with EXACTLY this structure: '
                        '{"optimized_query": "string"}'
                    ),
                },
                {
                    "role": "user",
                    "content": (
                        f"Target Tags: {state.get('target_tags', [])}\n\n"
                        f"User Query: {state['original_query']}"
                    ),
                },
            ],
        )

        content = response.choices[0].message.content
        parsed_json = json.loads(content)

        return {"expanded_query": parsed_json["optimized_query"]}

    except Exception as error:
        print(f"query_expander error: {error}")
        return {"expanded_query": state["original_query"]}


In [15]:
# =========================
# CELL 6: HyDE generator node
# =========================
import json

def hyde_generator(state: LabState) -> dict:
    try:
        response = groq_client.chat.completions.create(
            model="llama-3.1-8b-instant",
            response_format={"type": "json_object"},
            messages=[
                {
                    "role": "system",
                    "content": (
                        "You are a Material Science researcher writing a hypothetical excerpt for a peer-reviewed journal. "
                        "Your goal is to write a single, dense paragraph that incorporates the keywords from the user's query. "
                        "You are writing this so a vector database can use its mathematical structure to find similar real-world papers. "
                        "Do not worry about factual accuracy; focus entirely on mimicking the structural style, formatting, and advanced vocabulary of a real PVD research paper.\n\n"
                        "Tag guidance:\n"
                        "- If the target tags include 'Background', write it like an 'Introduction' section.\n"
                        "- If the tags include 'Synthesis', write the paragraph in the style of an 'Experimental Methods' section.\n"
                        "- If the tags include 'Characterization', write it like a 'Materials Characterization' section.\n"
                        "- If the tags include 'Analysis', write it like a 'Results and Discussion' section.\n"
                        "- If multiple tags, blend the styles smoothly.\n\n"
                        "Additional rules:\n"
                        "- Write exactly one dense paragraph.\n"
                        "- Use formal academic tone and domain-specific vocabulary.\n"
                        "- Do not add bullet points, lists, headings, or explanations.\n"
                        "- Do not mention that the paragraph is hypothetical.\n\n"
                        'Return strict JSON with EXACTLY this structure: '
                        '{"hypothetical_document": "string"}'
                    ),
                },
                {
                    "role": "user",
                    "content": (
                        f"Target Tags: {state.get('target_tags', [])}\n\n"
                        f"Expanded Query: {state.get('expanded_query', '')}"
                    ),
                },
            ],
        )

        content = response.choices[0].message.content
        parsed_json = json.loads(content)

        return {"hyde_document": parsed_json["hypothetical_document"]}

    except Exception as error:
        print(f"hyde_generator error: {error}")
        return {"hyde_document": state.get("expanded_query", "")}


In [16]:
# =========================
# CELL 7: Load Chroma DB + reranker
# =========================
print("chromadb ok")
print("sentence-transformers ok")

def _resolve_chroma_path() -> str:
    working_path = "/kaggle/working/pvd_Category_chroma_db_final_embeddings"
    if os.path.exists(os.path.join(working_path, "chroma.sqlite3")):
        return working_path

    candidate_paths = []
    for root, dirs, files in os.walk("/kaggle/input"):
        if "chroma.sqlite3" in files:
            candidate_paths.append(root)

    if not candidate_paths:
        raise FileNotFoundError(
            "No Chroma persist directory was found. Run the classification notebook first or attach the persisted DB as a Kaggle dataset."
        )

    return candidate_paths[0]

CHROMA_PATH = _resolve_chroma_path()
print(f"Using Chroma path: {CHROMA_PATH}")

client = chromadb.PersistentClient(path=CHROMA_PATH)
pvd_collection = client.get_collection(name="pvd_docs")
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

print("Loaded collection: pvd_docs")


chromadb ok
sentence-transformers ok
Using Chroma path: /kaggle/working/pvd_Category_chroma_db_final_embeddings


config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

Loaded collection: pvd_docs


In [17]:
# =========================
# CELL 8: Retriever helpers
# =========================
def _build_tag_filter(director_categories: list[str]) -> dict | None:
    tag_to_field = {
        "Background": "is_Background",
        "Synthesis": "is_Synthesis",
        "Characterization": "is_Characterization",
        "Analysis": "is_Analysis",
    }

    filter_clauses = []
    for tag in director_categories:
        field_name = tag_to_field.get(tag)
        if field_name:
            filter_clauses.append({field_name: True})

    if not filter_clauses:
        return None

    if len(filter_clauses) == 1:
        return filter_clauses[0]

    return {"$or": filter_clauses}


In [18]:
# =========================
# CELL 9: Retriever node
# =========================
def retrieve_and_rerank(
    hyde_response: str,
    director_categories: list[str],
    top_k: int = 3,
) -> list[dict]:
    try:
        if not hyde_response or not isinstance(hyde_response, str):
            return []

        if not isinstance(director_categories, list):
            director_categories = []

        if not isinstance(top_k, int) or top_k <= 0:
            top_k = 3

        query_kwargs = {
            "query_texts": [hyde_response],
            "n_results": top_k * 3,
        }

        where_filter = _build_tag_filter(director_categories)
        if where_filter is not None:
            query_kwargs["where"] = where_filter

        results = pvd_collection.query(**query_kwargs)

        ids = results.get("ids", [[]])[0]
        documents = results.get("documents", [[]])[0]
        metadatas = results.get("metadatas", [[]])[0]

        if not ids or not documents:
            return []

        pairs = [[hyde_response, chunk_text] for chunk_text in documents]
        scores = cross_encoder.predict(pairs)

        reranked = []
        for doc_id, chunk_text, metadata, score in zip(ids, documents, metadatas, scores):
            reranked.append(
                {
                    "document_id": doc_id,
                    "text": chunk_text,
                    "score": float(score),
                    "doi": metadata.get("doi", ""),
                    "title": metadata.get("title", ""),
                    "chunk_idx": metadata.get("chunk_idx", None),
                }
            )

        reranked.sort(key=lambda item: item["score"], reverse=True)
        return reranked[:top_k]

    except Exception as error:
        print(f"retrieve_and_rerank error: {error}")
        return []


def retriever_node(state: LabState) -> dict:
    hyde_text = state.get("hyde_document", "")
    target_tags = state.get("target_tags", [])
    top_chunks = retrieve_and_rerank(hyde_text, target_tags, top_k=3)
    return {"retrieved_chunks": top_chunks}


In [19]:
# =========================
# CELL 9B: Query-expander retriever node
# =========================
def retrieve_and_rerank_from_expanded_query(
    expanded_query: str,
    director_categories: list[str],
    top_k: int = 3,
) -> list[dict]:
    try:
        if not expanded_query or not isinstance(expanded_query, str):
            return []

        if not isinstance(director_categories, list):
            director_categories = []

        if not isinstance(top_k, int) or top_k <= 0:
            top_k = 3

        query_kwargs = {
            "query_texts": [expanded_query],
            "n_results": top_k * 3,
        }

        where_filter = _build_tag_filter(director_categories)
        if where_filter is not None:
            query_kwargs["where"] = where_filter

        results = pvd_collection.query(**query_kwargs)

        ids = results.get("ids", [[]])[0]
        documents = results.get("documents", [[]])[0]
        metadatas = results.get("metadatas", [[]])[0]

        if not ids or not documents:
            return []

        pairs = [[expanded_query, chunk_text] for chunk_text in documents]
        scores = cross_encoder.predict(pairs)

        reranked = []
        for doc_id, chunk_text, metadata, score in zip(ids, documents, metadatas, scores):
            reranked.append(
                {
                    "document_id": doc_id,
                    "text": chunk_text,
                    "score": float(score),
                    "doi": metadata.get("doi", ""),
                    "title": metadata.get("title", ""),
                    "chunk_idx": metadata.get("chunk_idx", None),
                }
            )

        reranked.sort(key=lambda item: item["score"], reverse=True)
        return reranked[:top_k]

    except Exception as error:
        print(f"retrieve_and_rerank_from_expanded_query error: {error}")
        return []


def query_expander_retriever_node(state: LabState) -> dict:
    expanded_query = state.get("expanded_query", "")
    target_tags = state.get("target_tags", [])
    top_chunks = retrieve_and_rerank_from_expanded_query(expanded_query, target_tags, top_k=3)
    return {"expanded_query_chunks": top_chunks}


In [20]:
# =========================
# CELL 9C: Combine HyDE + Expanded Query results and rerank final top 3
# =========================
def combine_and_rerank_chunks(
    expanded_query: str,
    hyde_response: str,
    expanded_query_chunks: list[dict],
    hyde_chunks: list[dict],
    top_k: int = 3,
) -> list[dict]:
    try:
        if not isinstance(expanded_query_chunks, list):
            expanded_query_chunks = []

        if not isinstance(hyde_chunks, list):
            hyde_chunks = []

        if not isinstance(top_k, int) or top_k <= 0:
            top_k = 3

        combined = expanded_query_chunks + hyde_chunks
        if not combined:
            return []

        # Deduplicate by document_id and keep the better first-stage score
        deduped = {}
        for item in combined:
            doc_id = item["document_id"]
            if doc_id not in deduped or item["score"] > deduped[doc_id]["score"]:
                deduped[doc_id] = item

        unique_chunks = list(deduped.values())

        # Use both query views together for final reranking
        hybrid_query = f"{expanded_query}\n\n{hyde_response}".strip()
        pairs = [[hybrid_query, chunk["text"]] for chunk in unique_chunks]
        final_scores = cross_encoder.predict(pairs)

        final_reranked = []
        for chunk, score in zip(unique_chunks, final_scores):
            final_reranked.append(
                {
                    "document_id": chunk["document_id"],
                    "text": chunk["text"],
                    "score": float(score),
                    "doi": chunk.get("doi", ""),
                    "title": chunk.get("title", ""),
                    "chunk_idx": chunk.get("chunk_idx", None),
                }
            )

        final_reranked.sort(key=lambda item: item["score"], reverse=True)
        return final_reranked[:top_k]

    except Exception as error:
        print(f"combine_and_rerank_chunks error: {error}")
        return []


In [21]:
# =========================
# CELL 9D: Final hybrid retriever node
# =========================
def hybrid_retriever_node(state: LabState) -> dict:
    expanded_query = state.get("expanded_query", "")
    hyde_text = state.get("hyde_document", "")

    expanded_query_chunks = state.get("expanded_query_chunks", [])
    hyde_chunks = state.get("retrieved_chunks", [])

    final_top_chunks = combine_and_rerank_chunks(
        expanded_query=expanded_query,
        hyde_response=hyde_text,
        expanded_query_chunks=expanded_query_chunks,
        hyde_chunks=hyde_chunks,
        top_k=3,
    )

    return {"final_retrieved_chunks": final_top_chunks}


In [ ]:
# =========================
# CELL 10: End-to-end final retrieval test
# =========================
pipeline_test_queries = [
    "What depostion conditions for ZnO ?"
]

print("Running Director -> Expander -> HyDE -> Dual Retrieval -> Final Rerank test...\n" + "=" * 100)

for idx, query in enumerate(pipeline_test_queries, 1):
    print(f"\n[Test {idx}] Original Query: {query}")

    state: LabState = {
        "original_query": query,
        "route": "",
        "target_tags": [],
        "expanded_query": "",
        "hyde_document": "",
        "retrieved_chunks": [],
        "expanded_query_chunks": [],
        "final_retrieved_chunks": [],
    }

    # Director
    state.update(chief_director(state))
    print(f"   -> [Director] Route: {state['route'].upper()}")
    print(f"   -> [Director] Tags: {state.get('target_tags', [])}")

    if state["route"] == "database":
        # Query Expander
        state.update(query_expander(state))
        print(f"   -> [Expander] Expanded Query:\n      {state['expanded_query']}")

        # HyDE
        state.update(hyde_generator(state))
        print(f"   -> [HyDE] Hypothetical Paragraph:\n      {state['hyde_document'][:500]}...")

        # Retrieval from expanded query
        state.update(query_expander_retriever_node(state))
        print("   -> [Expanded Query Retriever] Top Chunks:")
        if not state["expanded_query_chunks"]:
            print("      No chunks returned.")
        else:
            for rank, chunk in enumerate(state["expanded_query_chunks"], 1):
                print(f"      [{rank}] score={chunk['score']:.4f} | doi={chunk.get('doi', '')} | title={chunk.get('title', '')}")

        # Retrieval from HyDE
        state.update(retriever_node(state))
        print("   -> [HyDE Retriever] Top Chunks:")
        if not state["retrieved_chunks"]:
            print("      No chunks returned.")
        else:
            for rank, chunk in enumerate(state["retrieved_chunks"], 1):
                print(f"      [{rank}] score={chunk['score']:.4f} | doi={chunk.get('doi', '')} | title={chunk.get('title', '')}")

        # Final combined rerank
        state.update(hybrid_retriever_node(state))
        print("   -> [Final Hybrid Retriever] Final Top 3 Chunks:")
        if not state["final_retrieved_chunks"]:
            print("      No final chunks returned.")
        else:
            for rank, chunk in enumerate(state["final_retrieved_chunks"], 1):
                print(f"      [{rank}] FINAL score={chunk['score']:.4f} | doi={chunk.get('doi', '')} | title={chunk.get('title', '')}")
                preview = chunk.get("text", "").replace("\n", " ")
                print(f"          {preview[:300]}...")

    else:
        print("   -> [Pipeline] Skipped database branch.")

    if idx < len(pipeline_test_queries):
        time.sleep(2)


In [22]:
!pip install -q -U google-generativeai

In [23]:
import json
from google import genai
from google.genai import types
from kaggle_secrets import UserSecretsClient
from typing import TypedDict, List


# =========================
# State
# =========================
class LabState(TypedDict):
    original_query: str
    route: str
    target_tags: List[str]
    expanded_query: str
    hyde_document: str
    expanded_query_chunks: List[dict]
    retrieved_chunks: List[dict]
    final_retrieved_chunks: List[dict]
    final_answer: str


# =========================
# Gemini setup: ONLY KEY 9
# =========================
user_secrets = UserSecretsClient()
gemini_api_key = user_secrets.get_secret("GEMINI_API_KEY_8")
gemini_client = genai.Client(api_key=gemini_api_key)

MODEL_NAME = "gemini-3-flash-preview"


# =========================
# Helper: build context from top 3 chunks
# =========================
def build_llm_context(final_retrieved_chunks: list[dict]) -> str:
    if not final_retrieved_chunks:
        return "No retrieved evidence available."

    context_blocks = []
    for idx, chunk in enumerate(final_retrieved_chunks[:3], 1):
        context_blocks.append(
            "\n".join(
                [
                    f"Chunk {idx}",
                    f"DOI: {chunk.get('doi', '')}",
                    f"Title: {chunk.get('title', '')}",
                    f"Chunk Index: {chunk.get('chunk_idx', '')}",
                    f"Text: {chunk.get('text', '')}",
                ]
            )
        )

    return "\n\n".join(context_blocks)


# =========================
# Helper: build final answer prompt
# =========================
def build_final_answer_prompt(user_query: str, retrieved_chunks: list[dict]) -> str:
    evidence_context = build_llm_context(retrieved_chunks)

    return f"""
You are a Materials Science AI Copilot for Physical Vapor Deposition (PVD) research.

Your task:
Answer the user’s question strictly using the retrieved chunks as the only source of truth.

Rules:
Use only the retrieved chunks to construct your answer.
Do not use prior knowledge, assumptions, or external information beyond the provided chunks.
If the answer is fully supported by the chunks, respond clearly and cite evidence inline using:
[Chunk 1], [Chunk 2], [Chunk 3], etc.
If the chunks provide partial information, answer only the supported portion and do not fill gaps.


Do not infer, speculate, or generalize beyond what is explicitly stated in the chunks.
Do not fabricate citations or add unsupported claims.
Keep the response precise, direct, and academically structured.

User Query:
{user_query}

Retrieved Evidence:
{evidence_context}

Return only the final answer text.
""".strip()


# =========================
# Final answer generator
# =========================
def generate_final_answer(user_query: str, final_retrieved_chunks: list[dict]) -> str:
    try:
        prompt = build_final_answer_prompt(user_query, final_retrieved_chunks)

        response = gemini_client.models.generate_content(
            model=MODEL_NAME,
            contents=prompt,
            config=types.GenerateContentConfig(
                temperature=0.2,
            ),
        )

        return response.text.strip()

    except Exception as error:
        print(f"generate_final_answer error: {error}")
        return "I could not generate a final answer from the retrieved chunks."


# =========================
# LangGraph-style final answer node
# =========================
def final_answer_node(state: LabState) -> dict:
    user_query = state.get("original_query", "")
    final_chunks = state.get("final_retrieved_chunks", [])

    answer = generate_final_answer(user_query, final_chunks)
    return {"final_answer": answer}


In [24]:
def final_paraphrase_node(state: dict) -> dict:
    try:
        original_query = state.get("original_query", "").strip()
        draft_answer = state.get("final_answer", "").strip()

        if not draft_answer:
            return {}

        tokenizer, model = load_paraphrase_model()

        prompt = (
             f"You are a scientific paraphrasing assistant.\n"
             f"Rewrite the following answer using completely different wording.\n"
             f"Remove all inline citations like [Chunk 1], [Chunk 2], etc. from the main body.\n"
             f"Keep all scientific facts, numbers, and units exactly as they are.\n"
             f"After the answer, add a final section exactly titled:\n"
             f"Citations :-\n"
             f"------------\n"
             f"In that section, reproduce only the citation labels that were present in the original draft answer, one per line.\n"
             f"Do not invent new citations.\n"
             f"Do not place citations inside the main paragraph text.\n\n"
             f"Query: {original_query}\n\n"
             f"Answer to paraphrase:\n{draft_answer}\n\n"
             f"Paraphrased answer:"
        )
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=256,
                temperature=0.7,
                do_sample=True,
                top_p=0.9,
                repetition_penalty=1.05,
                pad_token_id=tokenizer.eos_token_id,
            )

        generated_ids = outputs[0][inputs["input_ids"].shape[1]:]
        paraphrased = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()

        if not paraphrased:
            paraphrased = draft_answer

        return {
            "final_answer_raw": draft_answer,
            "final_answer": paraphrased,
        }

    except Exception as e:
        print(f"final_paraphrase_node error: {e}")
        return {
            "final_answer_raw": state.get("final_answer", ""),
            "final_answer": state.get("final_answer", ""),
        }


In [30]:
test_query = "What are modern characterization techniques in Pulsed Layer Deposition ?"

state: LabState = {
    "original_query": test_query,
    "route": "",
    "target_tags": [],
    "expanded_query": "",
    "hyde_document": "",
    "expanded_query_chunks": [],
    "retrieved_chunks": [],
    "final_retrieved_chunks": [],
    "final_answer": "",
    "final_answer_raw": "",
}


state.update(chief_director(state))
state.update(query_expander(state))
state.update(hyde_generator(state))
state.update(query_expander_retriever_node(state))
state.update(retriever_node(state))
state.update(hybrid_retriever_node(state))
state.update(final_answer_node(state))
state.update(final_paraphrase_node(state))   

print("QUERY:")
print(state["original_query"])

"""print("\nFINAL TOP 3 CHUNKS:")
for i, chunk in enumerate(state["final_retrieved_chunks"], 1):
    print(f"\n[{i}] score={chunk['score']:.4f}")
    print("DOI:", chunk.get("doi", ""))
    print("Title:", chunk.get("title", ""))
    print("Text:", chunk.get("text", "")[:400], "...")
    
print("\nFINAL ANSWER Raw:")
print(state["final_answer_raw"])"""

print("\nFINAL ANSWER:")
print(state["final_answer"])

QUERY:
What are modern characterization techniques in Pulsed Layer Deposition ?

FINAL ANSWER:
Based on the provided research, modern characterization techniques used in Pulsed Laser Deposition (PLD) and related technologies include:

*   Surface Morphology and Topography: Atomic Force Microscopy (AFM) is used to acquire images of surface morphology, characterize surface topography, and measure surface roughness (including root mean square roughness, $S_q$). AFM imaging is also utilized for nanoparticle size analysis.
*   Thickness Measurement: Layer thickness is measured using an Alfa-step profilometer or a Wyko optical profilometer.
*   Chemical and Structural Analysis:
    *   ATR-FTIR: Attenuated Total Reflection Fourier Transform Infrared spectroscopy is used to analyze chemical composition and check for residual solvents.
    *   Laser Micro-Raman Spectroscopy (MRS): This technique is used to investigate film structure and can generate Raman maps of the surface by measuring scatt

In [ ]:
!pip install -q "chromadb==0.5.5" "sentence-transformers<4.0.0" pandas openpyxl


In [ ]:
pip install --upgrade chromadb posthog

In [ ]:
!pip uninstall -y chromadb
!pip install -q "chromadb==0.5.5" "sentence-transformers<4.0.0" pandas openpyxl


In [5]:
import json
import os
import shutil
import pandas as pd
import chromadb
from sentence_transformers import SentenceTransformer
import torch

# ==============================
# PATHS
# ==============================
CSV_PATH = "/kaggle/input/datasets/aads19/pld-categorized-final/PLD CATEGORY FINAL DATASET.csv"
CHROMA_DIR = "/kaggle/working/pvd_Category_chroma_db_final_embeddings"

# ==============================
# CHECK GPU
# ==============================
print("GPU Available:", torch.cuda.is_available())

# ==============================
# CLEAN START
# ==============================
if os.path.exists(CHROMA_DIR):
    shutil.rmtree(CHROMA_DIR)

# ==============================
# LOAD MODEL (GPU)
# ==============================
model = SentenceTransformer("BAAI/bge-small-en-v1.5", device="cuda")

# ==============================
# CREATE CHROMA CLIENT
# ==============================
chroma_client = chromadb.PersistentClient(path=CHROMA_DIR)

# ⚠️ No embedding function
pvd_collection = chroma_client.get_or_create_collection(
    name="pvd_docs"
)

print("Starting Chroma rebuild...")

# ==============================
# LOAD CSV
# ==============================
df = pd.read_csv(CSV_PATH)

documents = []
metadatas = []
ids = []

for idx, row in df.iterrows():
    try:
        tag_dict = json.loads(row["tags"])
        chunk_tags = tag_dict.get("tags", [])
    except Exception:
        chunk_tags = []

    metadata = {
        "doi": str(row["doi"]),
        "title": str(row["title"]),
        "chunk_idx": int(row["chunk_start_idx"]),
        "is_Background": "Background" in chunk_tags,
        "is_Synthesis": "Synthesis" in chunk_tags,
        "is_Characterization": "Characterization" in chunk_tags,
        "is_Analysis": "Analysis" in chunk_tags,
    }

    documents.append(str(row["text_chunk"]))
    metadatas.append(metadata)
    ids.append(f"{row['doi']}_chunk_{idx}")

# ==============================
# GENERATE EMBEDDINGS (GPU)
# ==============================
print("Generating embeddings...")

EMBED_BATCH_SIZE = 256
embeddings = []

for i in range(0, len(documents), EMBED_BATCH_SIZE):
    batch = documents[i:i+EMBED_BATCH_SIZE]

    emb = model.encode(
        batch,
        batch_size=256,
        show_progress_bar=True,
        convert_to_numpy=True
    )

    embeddings.extend(emb)
    print(f"Embedded {min(i+EMBED_BATCH_SIZE, len(documents))}/{len(documents)}")

# ==============================
# STORE IN CHROMA (BATCHED)
# ==============================
print("Storing in Chroma DB...")

ADD_BATCH_SIZE = 1000  # safe under Chroma limit

for i in range(0, len(documents), ADD_BATCH_SIZE):
    pvd_collection.add(
        documents=documents[i:i+ADD_BATCH_SIZE],
        metadatas=metadatas[i:i+ADD_BATCH_SIZE],
        ids=ids[i:i+ADD_BATCH_SIZE],
        embeddings=embeddings[i:i+ADD_BATCH_SIZE]
    )

    print(f"Stored {min(i+ADD_BATCH_SIZE, len(documents))}/{len(documents)}")

# ==============================
# VERIFY
# ==============================
print(f"\nTotal stored: {pvd_collection.count()}")

print(f"\n✅ Successfully rebuilt Chroma DB with {len(documents)} chunks.")
print(f"📁 Saved at: {CHROMA_DIR}")

GPU Available: True


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

Embedded 7936/8381


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embedded 8192/8381


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embedded 8381/8381
Storing in Chroma DB...
Stored 1000/8381
Stored 2000/8381
Stored 3000/8381
Stored 4000/8381
Stored 5000/8381
Stored 6000/8381
Stored 7000/8381
Stored 8000/8381
Stored 8381/8381

Total stored: 8381

✅ Successfully rebuilt Chroma DB with 8381 chunks.
📁 Saved at: /kaggle/working/pvd_Category_chroma_db_final_embeddings
